In [1]:
!pip install ultralytics opencv-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

cap = cv2.VideoCapture(0)

#해상도
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("LifeCam이 연결되었습니다. 'q'를 누르면 종료")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("카메라 프레임을 읽을 수 없음")
        break

    # classes=[0] 사람만 인식
    results = model(frame, stream=True, classes=0, conf=0.4)

    for r in results:
        annotated_frame = r.plot()

    cv2.imshow("LifeCam - YOLOv8 Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO('yolo26n-pose.pt')

# 2. LifeCam 연결 및 해상도 설정
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("손을 든 사람 인식 시작, 'q'를 누르면 종료.")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    results = model(frame, stream=True, conf=0.5)
    annotated_frame = frame.copy()

    for r in results:
        annotated_frame = r.plot()
        
        if r.keypoints is not None and len(r.keypoints.xy) > 0:
            for person_pts in r.keypoints.xy:
                if len(person_pts) > 10:
                    
                    # 왼쪽 관절 인덱스: 5(어깨), 7(팔꿈치), 9(손목)
                    left_shoulder_y = person_pts[5][1].item()
                    left_elbow_y = person_pts[7][1].item()
                    left_wrist_y = person_pts[9][1].item()
                    
                    # 오른쪽 관절 인덱스: 6(어깨), 8(팔꿈치), 10(손목)
                    right_shoulder_y = person_pts[6][1].item()
                    right_elbow_y = person_pts[8][1].item()
                    right_wrist_y = person_pts[10][1].item()
                    
                    # 0(머리관절)
                    head_y = person_pts[0][1].item()
                    
                    left_visible = (left_shoulder_y > 0 and left_elbow_y > 0 and left_wrist_y > 0)
                    right_visible = (right_shoulder_y > 0 and right_elbow_y > 0 and right_wrist_y > 0)
                    head_visible = (head_y > 0)
                    
                    is_lefthead_raised = head_visible and (left_wrist_y < head_y)
                    is_righthead_raised = head_visible and (right_wrist_y < head_y)
                    
                    is_left_hand_raised = left_visible and (left_wrist_y < left_shoulder_y) and (left_wrist_y < left_elbow_y)
                    is_right_hand_raised = right_visible and (right_wrist_y < right_shoulder_y) and (right_wrist_y < right_elbow_y)

                    # 왼쪽 팔이 확실히 들렸는가? (머리 위 또는 어깨 위)
                    left_up = is_left_hand_raised or is_lefthead_raised
                    
                    # 오른쪽 팔이 확실히 들렸는가?
                    right_up = is_right_hand_raised or is_righthead_raised

                    # 두 상태가 '다를 때만' (즉, 한쪽만 True일 때만) 실행!
                    if left_up != right_up:
                        
                        cv2.putText(annotated_frame, "One Hand Raised", (50, 100), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 5)
                        
                        print("손들엇다!!")

    cv2.imshow("Pose Detection - Smart Hand Raise", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

정교한 손들기 인식 모드를 시작합니다. 'q'를 누르면 종료됩니다.

0: 384x640 (no detections), 55.8ms
Speed: 1.5ms preprocess, 55.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 52.5ms
Speed: 1.5ms preprocess, 52.5ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 55.9ms
Speed: 1.4ms preprocess, 55.9ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 54.7ms
Speed: 1.4ms preprocess, 54.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 50.5ms
Speed: 1.3ms preprocess, 50.5ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 51.8ms
Speed: 1.4ms preprocess, 51.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 52.1ms
Speed: 1.8ms preprocess, 52.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 51